# UC2 - Return Fraud Risk Scoring + AI Investigation Briefs

#1) Installation 

In [0]:
%pip install -U openai

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


#2) Imports and Rule Weights

In [0]:
import json
import subprocess
import sys
import time
from datetime import datetime, timezone
from typing import List
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from openai import OpenAI

CATALOG = "globalmart.gold"
OUTPUT_TABLE = f"{CATALOG}.flagged_return_customers"
LLM_MODEL_NAME = "databricks-gpt-oss-20b"

# Risk threshold for investigation queue.
RISK_THRESHOLD = 40

# Rule weights (sum to 100).
W_R0 = 25  # no matching order (strongest signal)
W_R1 = 15
W_R2 = 15
W_R3 = 10
W_R4 = 10
W_R5 = 10
W_R6 = 15

## 3) Resolve source tables

In [0]:
def resolve_first_existing_table(candidates: List[str]) -> str:
    for name in candidates:
        if spark.catalog.tableExists(name):
            return name
    raise ValueError(f"None of these tables exist: {candidates}")


RETURNS_TABLE = resolve_first_existing_table([f"{CATALOG}.fact_return"])
CUSTOMERS_TABLE = resolve_first_existing_table([f"{CATALOG}.dim_customer"])
SALES_TABLE = resolve_first_existing_table([f"{CATALOG}.fact_sales"])

returns_df = spark.table(RETURNS_TABLE)
customers_df = spark.table(CUSTOMERS_TABLE)
sales_df = spark.table(SALES_TABLE)

# display(returns_df.limit(5))
# display(customers_df.limit(5))

## 4) Build customer return profile metrics
Rules are grounded in these measurable behaviors:
-   No matching order (return order_id not found in sales fact)
-   High return count (frequent returns by same customer)
-   High total return value (large cumulative refund exposure)
-   High average refund amount (high-value returns on average)
-   High rejected-return ratio (many returns fail validation)
-   Cross-region return activity (returns spread across multiple regions)
-   Concentrated return reason (same reason dominates return pattern)

In [0]:
base = (
    returns_df.join(
        customers_df.select(
            "customer_id",
            "customer_name",
            "customer_email",
            "customer_segment",
            "region_name",
        ),
        on="customer_id",
        how="left",
    )
    .join(
        sales_df.select("order_id").distinct().withColumn("order_found", F.lit(1)),
        on="order_id",
        how="left",
    )
    .withColumn("refund_amount", F.col("refund_amount").cast(DoubleType()))
    .withColumn("return_status_norm", F.lower(F.trim(F.col("return_status"))))
    .withColumn("return_reason_norm", F.lower(F.trim(F.col("return_reason"))))
)

overall = (
    base.groupBy("customer_id", "customer_name", "customer_email", "customer_segment", "region_name")
    .agg(
        F.count("*").alias("total_returns"),
        F.round(F.sum(F.coalesce(F.col("refund_amount"), F.lit(0.0))), 2).alias("total_return_value"),
        F.round(F.avg(F.coalesce(F.col("refund_amount"), F.lit(0.0))), 2).alias("avg_return_value"),
        F.sum(F.when(F.col("return_status_norm") == "rejected", 1).otherwise(0)).alias("rejected_returns"),
        F.sum(F.when(F.col("return_status_norm") == "approved", 1).otherwise(0)).alias("approved_returns"),
        F.countDistinct("region_key").alias("distinct_return_regions"),
        F.sum(F.when(F.col("order_found").isNull(), 1).otherwise(0)).alias("unmatched_returns"),
    )
    .withColumn(
        "rejected_ratio",
        F.when(F.col("total_returns") > 0, F.col("rejected_returns") / F.col("total_returns")).otherwise(F.lit(0.0)),
    )
    .withColumn(
        "unmatched_ratio",
        F.when(F.col("total_returns") > 0, F.col("unmatched_returns") / F.col("total_returns")).otherwise(F.lit(0.0)),
    )
)

reason_counts = (
    base.groupBy("customer_id", "return_reason_norm")
    .agg(F.count("*").alias("reason_count"))
)

top_reason = (
    reason_counts.withColumn(
        "rn",
        F.row_number().over(Window.partitionBy("customer_id").orderBy(F.desc("reason_count"))),
    )
    .filter(F.col("rn") == 1)
    .select(
        "customer_id",
        F.col("return_reason_norm").alias("top_return_reason"),
        F.col("reason_count").alias("top_reason_count"),
    )
)

profile = (
    overall.join(top_reason, on="customer_id", how="left")
    .withColumn(
        "top_reason_share",
        F.when(F.col("total_returns") > 0, F.col("top_reason_count") / F.col("total_returns")).otherwise(F.lit(0.0)),
    )
)

# display(profile.limit(20))

## 5) Apply weighted anomaly rules (0-100)
Chosen weights (defensible business signals):
 - R0 no_matching_order (unmatched_returns >= 1) -> 25
 - R1 high_return_count (>= 4) -> 15
 - R2 high_total_return_value (>= 1000) -> 15
 - R3 high_avg_refund (>= 200) -> 10
 - R4 high_rejected_ratio (>= 0.50 and total_returns>=4) -> 10
 - R5 cross_region_returns (distinct_return_regions >= 3) -> 10
 - R6 repeated_same_reason (top_reason_share >= 0.70 and total_returns>=5) -> 15

In [0]:
scored = (
    profile.withColumn("r0_no_matching_order", (F.col("unmatched_returns") >= 1).cast(IntegerType()))
    .withColumn("r1_high_return_count", (F.col("total_returns") >= 4).cast(IntegerType()))
    .withColumn("r2_high_total_value", (F.col("total_return_value") >= 1000).cast(IntegerType()))
    .withColumn("r3_high_avg_refund", (F.col("avg_return_value") >= 200).cast(IntegerType()))
    .withColumn(
        "r4_high_rejected_ratio",
        ((F.col("rejected_ratio") >= 0.50) & (F.col("total_returns") >= 4)).cast(IntegerType()),
    )
    .withColumn("r5_cross_region", (F.col("distinct_return_regions") >= 3).cast(IntegerType()))
    .withColumn(
        "r6_repeat_reason",
        ((F.col("top_reason_share") >= 0.70) & (F.col("total_returns") >= 5)).cast(IntegerType()),
    )
    .withColumn(
        "anomaly_score",
        (
            F.col("r0_no_matching_order") * F.lit(W_R0)
            + F.col("r1_high_return_count") * F.lit(W_R1)
            + F.col("r2_high_total_value") * F.lit(W_R2)
            + F.col("r3_high_avg_refund") * F.lit(W_R3)
            + F.col("r4_high_rejected_ratio") * F.lit(W_R4)
            + F.col("r5_cross_region") * F.lit(W_R5)
            + F.col("r6_repeat_reason") * F.lit(W_R6)
        ).cast(IntegerType()),
    )
    .withColumn(
        "rules_violated",
        F.concat_ws(
            ";",
            F.when(F.col("r0_no_matching_order") == 1, F.lit("R0_no_matching_order")),
            F.when(F.col("r1_high_return_count") == 1, F.lit("R1_high_return_count")),
            F.when(F.col("r2_high_total_value") == 1, F.lit("R2_high_total_return_value")),
            F.when(F.col("r3_high_avg_refund") == 1, F.lit("R3_high_avg_refund")),
            F.when(F.col("r4_high_rejected_ratio") == 1, F.lit("R4_high_rejected_ratio")),
            F.when(F.col("r5_cross_region") == 1, F.lit("R5_cross_region_returns")),
            F.when(F.col("r6_repeat_reason") == 1, F.lit("R6_repeated_return_reason")),
        ),
    )
    .withColumn("is_flagged", F.col("anomaly_score") >= F.lit(RISK_THRESHOLD))
)

flagged = scored.filter(F.col("is_flagged"))
# display(flagged.orderBy(F.desc("anomaly_score")).limit(100))

## 6) Databricks OpenAI-compatible client

In [0]:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
_ws_raw = spark.conf.get("spark.databricks.workspaceUrl")
_ws_url = _ws_raw.replace("https://", "").replace("http://", "").strip("/")
client = OpenAI(api_key=DATABRICKS_TOKEN, base_url=f"https://{_ws_url}/serving-endpoints")


## 7) Generate AI investigation brief for each flagged customer

In [0]:

BRIEF_PROMPT = """You are a return-fraud investigation assistant for GlobalMart.
Write a concise investigation brief (6-8 sentences) for the returns team.

Customer profile:
- customer_id: {customer_id}
- customer_name: {customer_name}
- customer_email: {customer_email}
- customer_segment: {customer_segment}
- home_region: {region_name}
- total_returns: {total_returns}
- total_return_value: {total_return_value}
- avg_return_value: {avg_return_value}
- rejected_returns: {rejected_returns}
- approved_returns: {approved_returns}
- rejected_ratio: {rejected_ratio}
- unmatched_returns: {unmatched_returns}
- unmatched_ratio: {unmatched_ratio}
- distinct_return_regions: {distinct_return_regions}
- top_return_reason: {top_return_reason}
- top_reason_share: {top_reason_share}
- anomaly_score: {anomaly_score}
- rules_violated: {rules_violated}

Your brief MUST include:
1) What specific patterns are suspicious (cite actual values above),
2) What innocent explanations could exist,
3) What the team should verify first with concrete steps.
Do not mention Python, Spark, or model internals."""


def call_llm_brief(prompt: str) -> str:
    resp = client.chat.completions.create(
        model=LLM_MODEL_NAME,
        messages=[
            {"role": "system", "content": "You write practical fraud investigation briefs for retail operations."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1,
        max_tokens=700,
    )
    raw = resp.choices[0].message.content if resp.choices and resp.choices[0].message else ""
    if raw is None:
        return ""
    if isinstance(raw, str):
        return raw.strip()
    if isinstance(raw, list):
        out = []
        for x in raw:
            if isinstance(x, str) and x.strip():
                out.append(x.strip())
            elif isinstance(x, dict):
                t = x.get("text") or x.get("content")
                if isinstance(t, str) and t.strip():
                    out.append(t.strip())
        return "\n\n".join(out).strip()
    if isinstance(raw, dict):
        t = raw.get("text") or raw.get("content")
        if isinstance(t, str):
            return t.strip()
    return str(raw).strip()


rows = flagged.collect()
generated_at = datetime.now(timezone.utc)

out_rows = []
for r in rows:
    payload = r.asDict(recursive=True)
    prompt = BRIEF_PROMPT.format(
        customer_id=payload.get("customer_id"),
        customer_name=payload.get("customer_name"),
        customer_email=payload.get("customer_email"),
        customer_segment=payload.get("customer_segment"),
        region_name=payload.get("region_name"),
        total_returns=payload.get("total_returns"),
        total_return_value=payload.get("total_return_value"),
        avg_return_value=payload.get("avg_return_value"),
        rejected_returns=payload.get("rejected_returns"),
        approved_returns=payload.get("approved_returns"),
        rejected_ratio=payload.get("rejected_ratio"),
        unmatched_returns=payload.get("unmatched_returns"),
        unmatched_ratio=payload.get("unmatched_ratio"),
        distinct_return_regions=payload.get("distinct_return_regions"),
        top_return_reason=payload.get("top_return_reason"),
        top_reason_share=payload.get("top_reason_share"),
        anomaly_score=payload.get("anomaly_score"),
        rules_violated=payload.get("rules_violated"),
    )
    brief = call_llm_brief(prompt)
    time.sleep(0.3)
    out_rows.append(
        (
            payload.get("customer_id"),
            payload.get("customer_name"),
            payload.get("customer_email"),
            payload.get("customer_segment"),
            payload.get("region_name"),
            int(payload.get("total_returns") or 0),
            float(payload.get("total_return_value") or 0.0),
            float(payload.get("avg_return_value") or 0.0),
            int(payload.get("rejected_returns") or 0),
            int(payload.get("approved_returns") or 0),
            float(payload.get("rejected_ratio") or 0.0),
            int(payload.get("unmatched_returns") or 0),
            float(payload.get("unmatched_ratio") or 0.0),
            int(payload.get("distinct_return_regions") or 0),
            payload.get("top_return_reason"),
            float(payload.get("top_reason_share") or 0.0),
            int(payload.get("anomaly_score") or 0),
            payload.get("rules_violated"),
            brief,
            generated_at,
        )
    )


## 8) Write `globalmart.gold.flagged_return_customers`

In [0]:

schema_out = StructType(
    [
        StructField("customer_id", StringType(), False),
        StructField("customer_name", StringType(), True),
        StructField("customer_email", StringType(), True),
        StructField("customer_segment", StringType(), True),
        StructField("region_name", StringType(), True),
        StructField("total_returns", IntegerType(), False),
        StructField("total_return_value", DoubleType(), False),
        StructField("avg_return_value", DoubleType(), False),
        StructField("rejected_returns", IntegerType(), False),
        StructField("approved_returns", IntegerType(), False),
        StructField("rejected_ratio", DoubleType(), False),
        StructField("unmatched_returns", IntegerType(), False),
        StructField("unmatched_ratio", DoubleType(), False),
        StructField("distinct_return_regions", IntegerType(), False),
        StructField("top_return_reason", StringType(), True),
        StructField("top_reason_share", DoubleType(), False),
        StructField("anomaly_score", IntegerType(), False),
        StructField("rules_violated", StringType(), True),
        StructField("ai_investigation_brief", StringType(), True),
        StructField("generated_at", TimestampType(), False),
    ]
)

out_df = spark.createDataFrame(out_rows, schema=schema_out)
(out_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE))

print(f"Wrote {out_df.count()} flagged customers to {OUTPUT_TABLE}")

Wrote 29 flagged customers to globalmart.gold.flagged_return_customers


## 9) Validate output

In [0]:
display(spark.table(OUTPUT_TABLE).orderBy(F.desc("anomaly_score")))

customer_id,customer_name,customer_email,customer_segment,region_name,total_returns,total_return_value,avg_return_value,rejected_returns,approved_returns,rejected_ratio,unmatched_returns,unmatched_ratio,distinct_return_regions,top_return_reason,top_reason_share,anomaly_score,rules_violated,ai_investigation_brief,generated_at
KH-16690,Kristen Hastings,bancboy@icloud.com,Corporate,West,5,1365.51,273.1,1,2,0.2,1,0.2,1,defective product,0.4,65,R0_no_matching_order;R1_high_return_count;R2_high_total_return_value;R3_high_avg_refund,"**Investigation Brief – Customer KH‑16690 (Kristen Hastings)** 1. The customer has 5 returns totaling $1,365.51, with an average return value of $273.10, which is 2.5× higher than the corporate segment average. 2. Two returns were approved while one was rejected, giving a 20 % rejection ratio, and one return remains unmatched, also a 20 % unmatched ratio—both above the typical 5–10 % thresholds. 3. All returns originate from a single West region store, yet the top reason “defective product” accounts for only 40 % of the returns, leaving 60 % unexplained. 4. The anomaly score of 65 and the violation of four rules (R0, R1, R2, R3) flag a potential return‑fraud scenario. 5. Innocent explanations could include a faulty product line released in that region, a clerical error in matching orders, or a legitimate high‑value purchase that was mistakenly returned. 6. Verify first by: (a) cross‑checking each return against the original order IDs and shipping dates; (b) reviewing the product batch and defect reports for the West region; (c) confirming the refund amounts against the original payment records; and (d) contacting the customer to confirm the reasons for each return. 7. If discrepancies persist after these checks, proceed with a deeper audit of the return authorization workflow and consider flagging the account for further monitoring.",2026-03-27T11:22:37.493Z
BP-11185,Ben Peterman,bancboy@icloud.com,Corporate,West,7,2057.47,293.92,1,1,0.14285714285714285,0,0.0,1,not satisfied,0.7142857142857143,55,R1_high_return_count;R2_high_total_return_value;R3_high_avg_refund;R6_repeated_return_reason,"**Investigation Brief – BP‑11185 (Ben Peterman)** Ben has made 7 returns totaling $2,057.47, an average of $293.92 per return, with 1 rejection (14.3 % rejection ratio). The top reason, “not satisfied,” accounts for 71.4 % of his returns, and the anomaly score is 55, triggering R1 (high return count), R2 (high total return value), R3 (high average refund), and R6 (repeated return reason). These metrics together flag a pattern of frequent, high‑value returns from a single corporate customer in the West region. Possible innocent explanations include legitimate corporate inventory adjustments, seasonal product returns, or a systematic issue with a specific product line that caused repeated dissatisfaction. Another scenario is that the returns were processed by a single employee or department, leading to a cluster of similar reasons. To verify, the team should: 1) review the original purchase orders for each return to confirm product details and shipping addresses; 2) cross‑check the return timestamps against the customer’s order history to detect any clustering or timing anomalies; 3) confirm that the return reasons match the original product issues documented in the order notes; 4) verify that the same employee or location handled all returns; and 5) compare the return amounts against the average refund for the product category to ensure they are not outliers.",2026-03-27T11:22:37.493Z
CK-12205,Chloris Kastensmidt,cremonini@me.com,Consumer,East,4,1165.07,291.27,2,0,0.5,0,0.0,1,wrong delivery,0.25,50,R1_high_return_count;R2_high_total_return_value;R3_high_avg_refund;R4_high_rejected_ratio,"**Investigation Brief – CK‑12205 (Chloris Kastensmidt)** 1. The customer has 4 returns totaling $1,165.07, an average of $291.27 per return, with 2 of those (50 %) rejected. These figures trigger rules R1–R4: high return count, high tota